# Project - Airline AI Assistant (Migrated to Google Gemini)

## Migration du cours LLM Engineering : OpenAI → Google Gemini
- Chat : GPT-4o-mini          → gemini-2.0-flash
- Images : DALL-E-3           → Imagen 3 (imagen-3.0-generate-002)
- TTS : openai tts-1          → gTTS (Google Text-to-Speech, gratuit)
- Function calling            → google.genai types.Tool


In [12]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import json
from dotenv import load_dotenv
from google import genai
from google.genai import types
import gradio as gr

# Pour l'audio
import base64
from io import BytesIO
from PIL import Image
from gtts import gTTS
from IPython.display import Audio, display

In [36]:
# ── Initialisation ────────────────────────────────────────────────────────────
load_dotenv(override=True)

gemini_api_key = os.getenv('GEMINI_API_KEY')
if gemini_api_key:
    print(f"Gemini API Key trouvée, commence par : {gemini_api_key[:8]}")
else:
    print("⚠️  GEMINI_API_KEY non définie dans .env")

Gemini API Key trouvée, commence par : AIzaSyBo


In [ ]:
client = genai.Client(api_key=gemini_api_key)

MODEL       = "gemini-2.0-flash"
IMAGE_MODEL = "imagen-3.0-generate-002"

In [47]:
system_message = (
    "You are a helpful assistant for an Airline called FlightAI. "
    "Give short, courteous answers, no more than 1 sentence. "
    "Always be accurate. If you don't know the answer, say so."
)

In [48]:
# ── Données & outil métier ────────────────────────────────────────────────────
ticket_prices = {
    "london": "$799",
    "paris":  "$899",
    "tokyo":  "$1400",
    "berlin": "$499",
}

def get_ticket_price(destination_city: str) -> str:
    """Retourne le prix d'un aller-retour vers destination_city."""
    print(f"Tool get_ticket_price called for {destination_city}")
    return ticket_prices.get(destination_city.lower(), "Unknown")


In [49]:
# ── Déclaration de l'outil pour Gemini ───────────────────────────────────────
# google-genai accepte directement une fonction Python annotée
price_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="get_ticket_price",
            description=(
                "Get the price of a return ticket to the destination city. "
                "Call this whenever you need to know the ticket price, for example "
                "when a customer asks 'How much is a ticket to this city'."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "destination_city": types.Schema(
                        type=types.Type.STRING,
                        description="The city that the customer wants to travel to",
                    )
                },
                required=["destination_city"],
            ),
        )
    ]
)


In [50]:
# ── Gestion du tool call renvoyé par Gemini ───────────────────────────────────
def handle_tool_call(function_call):
    """
    Reçoit un objet FunctionCall de Gemini,
    exécute la fonction locale et retourne un FunctionResponse.
    """
    city  = function_call.args.get("destination_city", "")
    price = get_ticket_price(city)

    tool_response = types.Part.from_function_response(
        name=function_call.name,
        response={"destination_city": city, "price": price},
    )
    return tool_response, city

In [51]:
# ── Génération d'image avec Imagen 3 ─────────────────────────────────────────
def artist(city: str) -> Image.Image:
    """
    Génère une image touristique de la ville avec Imagen 3.
    ⚠️  Imagen 3 est disponible sur le Free Tier avec quota limité.
    """
    response = client.models.generate_images(
        model=IMAGE_MODEL,
        prompt=(
            f"An image representing a vacation in {city}, "
            f"showing tourist spots and everything unique about {city}, "
            "in a vibrant pop-art style"
        ),
        config=types.GenerateImagesConfig(number_of_images=1),
    )
    image_bytes = response.generated_images[0].image.image_bytes
    return Image.open(BytesIO(image_bytes))

In [52]:
# ── Text-to-Speech avec gTTS ─────────────────────────────────────────────────
def talker(message: str):
    """
    Synthèse vocale via gTTS (Google TTS, gratuit, sans API key).
    Nécessite : uv add gtts
    """
    tts = gTTS(text=message, lang="en")
    audio_buffer = BytesIO()
    tts.write_to_fp(audio_buffer)
    audio_buffer.seek(0)

    output_filename = "output_audio.mp3"
    with open(output_filename, "wb") as f:
        f.write(audio_buffer.read())

    display(Audio(output_filename, autoplay=True))

In [53]:
# ── Boucle de conversation principale ────────────────────────────────────────
def chat(history):
    """
    history : liste de dicts {"role": "user"|"assistant", "content": str}
    Retourne (history_mise_a_jour, image_ou_None)
    """
    # ── Reconstruction des messages pour Gemini ──────────────────────────────
    # Gemini attend une liste de types.Content avec role "user" ou "model"
    gemini_history = []
    for msg in history:
        role = "model" if msg["role"] == "assistant" else "user"

        # Gradio 6.x : content peut être une liste [{"text": "...", "type": "text"}]
        # ou directement une string selon la version
        raw = msg["content"]
        if isinstance(raw, list):
            text = " ".join(
                part["text"] for part in raw
                if isinstance(part, dict) and "text" in part
            )
        else:
            text = raw

        content = types.Content(
            role=role,
            parts=[types.Part.from_text(text=text)],
        )
        gemini_history.append(content)

    config = types.GenerateContentConfig(
        system_instruction=system_message,
        tools=[price_tool],
    )

    # ── Premier appel ────────────────────────────────────────────────────────
    response = client.models.generate_content(
        model=MODEL,
        contents=gemini_history,
        config=config,
    )

    image = None

    # ── Détection du tool call ───────────────────────────────────────────────
    candidate = response.candidates[0]
    function_call = None
    for part in candidate.content.parts:
        if part.function_call:
            function_call = part.function_call
            break

    if function_call:
        # Exécution locale de l'outil
        tool_response_part, city = handle_tool_call(function_call)

        # Génération de l'image
        image = artist(city)

        # Ajout du tour assistant (avec le function_call) + réponse outil
        gemini_history.append(candidate.content)
        gemini_history.append(
            types.Content(
                role="user",
                parts=[tool_response_part],
            )
        )

        # Second appel avec le résultat de l'outil
        response = client.models.generate_content(
            model=MODEL,
            contents=gemini_history,
            config=config,
        )

    # ── Extraction de la réponse textuelle ───────────────────────────────────
    reply = response.text

    history = history + [{"role": "assistant", "content": reply}]

    # TTS — commenter la ligne suivante pour désactiver l'audio
    talker(reply)

    return history, image

In [54]:
# ── Interface Gradio ──────────────────────────────────────────────────────────
with gr.Blocks() as ui:
    with gr.Row():
        chatbot      = gr.Chatbot(height=500)
        image_output = gr.Image(height=500)
    with gr.Row():
        entry = gr.Textbox(label="Chat with our FlightAI Assistant:")
    with gr.Row():
        clear = gr.Button("Clear")

    def do_entry(message, history):
        history = history + [{"role": "user", "content": message}]
        return "", history

    entry.submit(
        do_entry,
        inputs=[entry, chatbot],
        outputs=[entry, chatbot],
    ).then(
        chat,
        inputs=chatbot,
        outputs=[chatbot, image_output],
    )
    clear.click(lambda: None, inputs=None, outputs=chatbot, queue=False)

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


Tool get_ticket_price called for Tokyo


Traceback (most recent call last):
  File "/home/bedane/dev/llm_engeneering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/bedane/dev/llm_engeneering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/bedane/dev/llm_engeneering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2157, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/bedane/dev/llm_engeneering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1634, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/bedane/dev/llm_engeneering/.venv/lib/python3